### GOAL OF NOTEBOOK 05

We will:

* retrieve relevant airline complaints
* inject retrieved context into LLM
* generate intelligent grounded responses
* build true airline RAG assistant

THIS is where your AI becomes truly powerful.

In [1]:
# Imports
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from dotenv import load_dotenv
from openai import OpenAI

import os

In [2]:
# Load Environment Variables
load_dotenv("../.env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [3]:
# Initialize OpenRouter Client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

MODEL = "openai/gpt-oss-120b:free"

print("LLM initialized successfully!")

LLM initialized successfully!


In [4]:
# Load Processed Dataset
airline_df = pd.read_csv(
    "../data/processed/cleaned_airline_support.csv"
)

airline_df.head()

,airline,text,airline_sentiment,negativereason,clean_text,rag_document
0,Virgin America,@VirginAmerica plus you've added commercials t...,positive,No Issue,plus you ve added commercials to the experienc...,Airline: Virgin America\nSentiment: positive\n...
1,Virgin America,@VirginAmerica I didn't today... Must mean I n...,neutral,No Issue,i didn t today must mean i need to take anothe...,Airline: Virgin America\nSentiment: neutral\nI...
2,Virgin America,@VirginAmerica it's really aggressive to blast...,negative,Bad Flight,it s really aggressive to blast obnoxious ente...,Airline: Virgin America\nSentiment: negative\n...
3,Virgin America,@VirginAmerica and it's a really big bad thing...,negative,Can't Tell,and it s a really big bad thing about it,Airline: Virgin America\nSentiment: negative\n...
4,Virgin America,@VirginAmerica seriously would pay $30 a fligh...,negative,Can't Tell,seriously would pay 30 a flight for seats that...,Airline: Virgin America\nSentiment: negative\n...


In [5]:
# Load Embedding Model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [6]:
# Generate Embeddings
documents = airline_df["rag_document"].tolist()

embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)

print("Embeddings ready!")

Batches:   0%|          | 0/448 [00:00<?, ?it/s]

Embeddings ready!


In [7]:
# Retrieval Function
def retrieve_relevant_documents(query, top_k=3):

    query_embedding = embedding_model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    retrieved_docs = []

    for idx in top_indices:
        retrieved_docs.append(
            airline_df.iloc[idx]["rag_document"]
        )

    return retrieved_docs

### IMPORTANT

This function retrieves:

### relevant airline knowledge

before calling the LLM.

That is the HEART of RAG.

In [8]:
# Create RAG Prompt Function
def build_rag_prompt(user_query, retrieved_docs):

    context = "\n\n".join(retrieved_docs)

    prompt = f"""
You are an AI airline customer support assistant.

Use the retrieved airline complaint examples below
to help answer the user's question professionally.

Retrieved Knowledge:
{context}

User Question:
{user_query}

Provide a helpful customer support response.
"""

    return prompt

In [9]:
# Full RAG Pipeline
def airline_rag_agent(user_query):

    # retrieve documents
    retrieved_docs = retrieve_relevant_documents(user_query)

    # build prompt
    rag_prompt = build_rag_prompt(
        user_query,
        retrieved_docs
    )

    # call llm
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a professional airline support assistant."
            },
            {
                "role": "user",
                "content": rag_prompt
            }
        ],
        temperature=0.7,
        max_tokens=300
    )

    return response.choices[0].message.content

In [10]:
# FIRST REAL RAG TEST
response = airline_rag_agent(
    "My baggage was lost after landing. What should I do?"
)

print(response)

Hello! I’m sorry to hear that your baggage didn’t arrive with you. Let’s get this sorted as quickly as possible.

**1. Report the loss immediately**  
- **Go to the airline’s baggage service desk** (or its kiosk) at the airport.  
- Provide your **boarding pass**, **baggage claim tag** (the receipt you received when you checked the bag), and a **brief description of the bag** (size, color, brand, any distinguishing marks).  

**2. Get a written reference**  
- Ask the agent for a **Baggage Irregularity Report (BIR)** or a **reference number**.  
- Keep a copy of the report, the reference number, and any receipts you receive.  

**3. Verify contact information**  
- Confirm that the airline has your **phone number and email address** so they can reach you with updates.  

**4. Follow up**  
- Most airlines begin a trace within 24 hours. You can check the status online using the reference number or by calling the airline’s baggage‑services phone line.  
- If you haven’t heard back within